In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import datetime
import hashlib

In [9]:
services = ['yellow', 'green']
start_year = 2015
end_year = 2025
months = list(range(1,13))
run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

In [6]:
app = (
    SparkSession.builder
    .appName('01_ingesta_parquet_raw')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)


# credenciales spark
sf_option = {
    
}

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/30 22:06:28 WARN Utils: Your hostname, Andress-MacBook-Air-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.100.156 instead (on interface en0)
26/03/30 22:06:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/30 22:06:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
def get_nyc_url(service, year, month):
    return f"https://d37ci6vzurychx.cloudfront.net/trip-data/{service}_tripdata_{year}-{month:02d}.parquet"

In [12]:
def standardize_columns(df, target_columns):
    for col_name, col_type in target_columns.items():
        if col_name not in df.columns:
            df = df.withColumn(col_name, F.lit(None).cast(col_type))
    return df

In [8]:
def compute_trip_nk(df, service):
    columns = ['service_type', 'VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'total_amount']
    return df.withColumn('trip_nk', F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in columns]), 256))

In [10]:
def log_audit(service, year, month, status, rows_read=0, rows_written=0, notes=""):
    audit_row= [(run_id, service, year, month, status, datetime.datetime.now(), notes)]
    schema = "run_id string, service string, year int, month int, status string, rows_read long, rows_written long, event_timestamp timestamp, notes string"
    audit_df = app.createDataFrame(audit_row, schema)
    audit_df.write.format('snowflake').options(**sf_option).option('dbtable', 'load_audit').mode('append').save()

In [11]:
expected_schema = {
    'VendorID': IntegerType(),
    'tpep_pickup_datetime': TimestampType(),
    'tpep_dropoff_datetime': TimestampType(),
    'passenger_count': DoubleType(),
    'trip_distance': DoubleType(),
    'RatecodeID': DoubleType(),
    'PULocationID': IntegerType(),
    'DOLocationID': IntegerType(),
    'payment_type': LongType(),
    'fare_amount': DoubleType(),
    'total_amount': DoubleType()
}

In [ ]:
for service in services:
    for year in range(start_year, end_year + 1):
        for month in months:
            url = get_nyc_url(service, year, month)

            try: 
                df_raw = app.read.parquet(url)
                rows_read = df_raw.count()

                df_processed = standardize_columns(df_raw, expected_schema)

                df_processed = df_processed.withColumns({
                    "run_id": F.lit(run_id),
                    "service_type": F.lit(service),
                    "source_year": F.lit(year),
                    "source_month": F.lit(month),
                    "ingested_at_utc": F.current_timestamp(),
                    "source_path": F.lit(url)
                })

                df_processed = compute_trip_nk(df_processed, service)

                target_table = f'{service.upper()}_TRIPS'
                delete_query = f'Delete from raw.{target_table} where source_year = {year} and source_month = {month}'

                df_processed.write.format('snowflake').options(**sf_option).option('dbtable', target_table).mode('append').save()

                log_audit(run_id, service, year, month, "SUCCESS", rows_read, rows_read)
                print(f'Cargado: {service} {year}-{month}')
            except Exception as e:
                status = 'Missing' if '403' in str(e) or '404' in str(e) else 'ERROR'
                log_audit(run_id, service, year, month, status, 0, 0, notes=str(e)[:500])
                print(f'Error: {url} - {status}')


26/03/30 22:44:50 WARN FileSystem: Cannot load filesystem
java.util.ServiceConfigurationError: org.apache.hadoop.fs.FileSystem: Provider org.apache.hadoop.fs.viewfs.ViewFileSystem could not be instantiated
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:552)
	at java.base/java.util.ServiceLoader$ProviderImpl.newInstance(ServiceLoader.java:712)
	at java.base/java.util.ServiceLoader$ProviderImpl.get(ServiceLoader.java:672)
	at java.base/java.util.ServiceLoader$2.next(ServiceLoader.java:1256)
	at org.apache.hadoop.fs.FileSystem.loadFileSystems(FileSystem.java:3525)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3562)
	at org.apache.hadoop.fs.FsUrlStreamHandlerFactory.<init>(FsUrlStreamHandlerFactory.java:77)
	at org.apache.spark.sql.internal.SharedState$.liftedTree2$1(SharedState.scala:209)
	at org.apache.spark.sql.internal.SharedState$.org$apache$spark$sql$internal$SharedState$$setFsUrlStreamHandlerFactory(SharedState.scala:208)
	at org.apache.spark.

PySparkValueError: [FIELD_STRUCT_LENGTH_MISMATCH] Length of object (7) does not match with length of fields (9).

In [ ]:
df_audit_summary = app.read.format("snowflake").options(**sf_option) \
    .option("dbtable", "LOAD_AUDIT").load() \
    .filter(F.col("run_id") == run_id)

df_audit_summary.groupBy("service", "status").count().show()